# Entropy-Based OpenVPN Detection

This notebook detects OpenVPN traffic by calculating the **Shannon entropy** of UDP payloads.

## How it works
1. Read all packets from a PCAP file
2. Filter for UDP packets with raw payload data
3. Calculate Shannon entropy for each payload
4. Flag packets that look like encrypted VPN traffic (high entropy + large size)
5. If suspicious packets exceed a threshold, classify as OpenVPN

## Thresholds
- **Entropy > 7.5**: Encrypted data has entropy close to 8.0 (maximum). 7.5 is chosen to catch most encrypted traffic while minimizing false positives.
- **Packet size > 500**: OpenVPN data channel packets are typically large due to encryption overhead and MTU.
- **Port 1194**: Standard OpenVPN UDP port — any traffic on this port is flagged regardless of entropy.
- **Detection threshold**: At least 0.5% of total packets must be suspicious to classify the capture as containing OpenVPN.

In [ ]:
from scapy.all import rdpcap, UDP, Raw, IP
import math
import os

# Detection thresholds
ENTROPY_THRESHOLD = 7.5       # Shannon entropy threshold (max is 8.0 for random data)
PACKET_SIZE_THRESHOLD = 500   # Minimum packet size to consider (bytes)
OPENVPN_PORT = 1194           # Default OpenVPN UDP port
DETECTION_PERCENT_THRESHOLD = 0.5  # Minimum % of suspicious packets to trigger detection

def shannon_entropy(data):
    """Calculate Shannon entropy of a byte sequence."""
    if not data:
        return 0
    byte_freq = [0] * 256
    for b in data:
        byte_freq[b] += 1
    entropy = 0
    for f in byte_freq:
        if f:
            p = f / len(data)
            entropy -= p * math.log2(p)
    return entropy

def detect_openvpn(pcap_path):
    """Detect OpenVPN traffic in a PCAP file using entropy analysis."""
    packets = rdpcap(pcap_path)
    total_packets = len(packets)
    suspicious_count = 0
    suspicious_packets = []

    for i, pkt in enumerate(packets):
        if UDP in pkt and Raw in pkt and IP in pkt:
            udp = pkt[UDP]
            payload = bytes(pkt[Raw].load)
            entropy = shannon_entropy(payload)
            pkt_size = len(pkt)

            # Flag if: on OpenVPN port OR (high entropy AND large packet)
            if (
                udp.dport == OPENVPN_PORT or udp.sport == OPENVPN_PORT
                or (entropy > ENTROPY_THRESHOLD and pkt_size > PACKET_SIZE_THRESHOLD)
            ):
                suspicious_count += 1
                suspicious_packets.append({
                    'index': i,
                    'src': pkt[IP].src,
                    'dst': pkt[IP].dst,
                    'sport': udp.sport,
                    'dport': udp.dport,
                    'entropy': round(entropy, 4),
                    'size': pkt_size
                })

    detection_threshold_count = max(1, int((DETECTION_PERCENT_THRESHOLD / 100) * total_packets))

    print(f"\n[*] Total packets: {total_packets}")
    print(f"[*] Suspicious OpenVPN-like packets: {suspicious_count}")
    print(f"[*] Detection threshold ({DETECTION_PERCENT_THRESHOLD}%): {detection_threshold_count} packets")

    if suspicious_count >= detection_threshold_count:
        print("[+] OpenVPN traffic likely DETECTED in the pcap file.")
    else:
        print("[-] No OpenVPN traffic detected in the pcap file.")

    return suspicious_packets

## Analyze a PCAP File

The dataset path is relative to the project root. Modify the path below to test different captures.

In [ ]:
# Use relative path from project root
pcap_file = os.path.join("data", "VPN data sets", "43.204.236.255", "vpn traffic.pcap")

# Verify file exists
if not os.path.exists(pcap_file):
    print(f"ERROR: File not found: {pcap_file}")
    print("Make sure you have extracted 'VPN data sets.zip' into the 'data/' folder.")
else:
    suspicious = detect_openvpn(pcap_file)

    # Show first few suspicious packets
    if suspicious:
        print(f"\n--- First {min(5, len(suspicious))} suspicious packets ---")
        for sp in suspicious[:5]:
            print(f"  Pkt#{sp['index']}: {sp['src']}:{sp['sport']} -> {sp['dst']}:{sp['dport']}  "
                  f"entropy={sp['entropy']}, size={sp['size']}")